In [108]:
# Sistem Rekomendasi SPKLU Kalimantan Barat
### Metode Content-Based Filtering menggunakan TF-IDF dan Cosine Similarity

In [109]:
import pandas as pd
import ipywidgets as widgets

from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [110]:
df = pd.read_excel("tugasspklu.xlsx")
df.head()

,Nama Stasiun,Provinsi,Kota,TYPE CHARGE,KW,UP3,ULP
0,SPKLU PLN UID KALBAR,Kalimantan Barat,Kab. Kubu Raya,FAST CHARGING,50.0,21PTK,21130-SUNGAI RAYA
1,SPKLU PUTUSSIBAU,Kalimantan Barat,Kab. Kapuas Hulu,FAST CHARGING,30.0,21SGU,21330-PUTUSSIBAU
2,SPKLU PLN ULP SUKADANA,Kalimantan Barat,Kab. Kayong Utara,FAST CHARGING,30.0,21KTP,21410-SUKADANA
3,SPKLU HOTEL BORNEO,Kalimantan Barat,Kota Pontianak,FAST CHARGING,30.0,21PTK,21100-KOTA
4,SPKLU ULP SINTANG,Kalimantan Barat,Kab. Sintang,FAST CHARGING,50.0,21SGU,21310-SINTANG


In [111]:
#membuat kolom fitur
df["fitur"] = (
    df["Kota"].astype(str) + " " +
    df["TYPE CHARGE"].astype(str) + " " +
    df["KW"].astype(str)
)

df.head()


,Nama Stasiun,Provinsi,Kota,TYPE CHARGE,KW,UP3,ULP,fitur
0,SPKLU PLN UID KALBAR,Kalimantan Barat,Kab. Kubu Raya,FAST CHARGING,50.0,21PTK,21130-SUNGAI RAYA,Kab. Kubu Raya FAST CHARGING 50.0
1,SPKLU PUTUSSIBAU,Kalimantan Barat,Kab. Kapuas Hulu,FAST CHARGING,30.0,21SGU,21330-PUTUSSIBAU,Kab. Kapuas Hulu FAST CHARGING 30.0
2,SPKLU PLN ULP SUKADANA,Kalimantan Barat,Kab. Kayong Utara,FAST CHARGING,30.0,21KTP,21410-SUKADANA,Kab. Kayong Utara FAST CHARGING 30.0
3,SPKLU HOTEL BORNEO,Kalimantan Barat,Kota Pontianak,FAST CHARGING,30.0,21PTK,21100-KOTA,Kota Pontianak FAST CHARGING 30.0
4,SPKLU ULP SINTANG,Kalimantan Barat,Kab. Sintang,FAST CHARGING,50.0,21SGU,21310-SINTANG,Kab. Sintang FAST CHARGING 50.0


In [112]:
#TF-IDF
#mengubah teks menjadi vector.
tfidf = TfidfVectorizer()

tfidf_matrix = tfidf.fit_transform(df["fitur"])

print(tfidf_matrix.shape)

(63, 31)


In [113]:
#Cosine Similarity
#Seberapa mirip setiap SPKLU dengan SPKLU lainnya dalam bentuk matriks
similarity = cosine_similarity(tfidf_matrix)

print(similarity)


[[1.         0.19951217 0.19951217 ... 0.58953785 0.13124783 0.21669294]
 [0.19951217 1.         0.41863013 ... 0.18243784 0.1103057  0.18211705]
 [0.19951217 0.41863013 1.         ... 0.18243784 0.1103057  0.18211705]
 ...
 [0.58953785 0.18243784 0.18243784 ... 1.         0.12001559 0.39574125]
 [0.13124783 0.1103057  0.1103057  ... 0.12001559 1.         0.11980456]
 [0.21669294 0.18211705 0.18211705 ... 0.39574125 0.11980456 1.        ]]


In [114]:
#Mengambil 5 SPKLU paling mirip (Rekomendasi)
def rekomendasi(nama_spklu):

    index = df[df["Nama Stasiun"] == nama_spklu].index[0]

    skor = list(enumerate(similarity[index]))

    skor = sorted(skor, key=lambda x: x[1], reverse=True)

    skor = skor[1:6]

    print("Rekomendasi SPKLU\n")

    for i, nilai in skor:

        print(df.iloc[i]["Nama Stasiun"],
              "- Similarity:",
              round(nilai,3))

In [115]:
dropdown = widgets.Dropdown(
    options=df["Nama Stasiun"].tolist(),
    description="Pilih:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px")
)

display(dropdown)

Dropdown(description='Pilih:', layout=Layout(width='500px'), options=('SPKLU PLN UID KALBAR', 'SPKLU PUTUSSIBA…

In [116]:
button = widgets.Button(
    description="Tampilkan Rekomendasi",
    button_style="success"
)
display(button)

Button(button_style='success', description='Tampilkan Rekomendasi', style=ButtonStyle())

Anda Memilih :  SPKLU Hotel Mercure Pontianak
Rekomendasi SPKLU

SPKLU HOTEL MAESTRO PONTIANAK - Similarity: 1.0
SPKLU PLN UP3 PONTIANAK - Similarity: 1.0
SPKLU PLN ULP SEI JAWI - Similarity: 1.0
SPKLU PLN ULP SIANTAN - Similarity: 1.0
SPKLU BNI PONTIANAK - Similarity: 1.0


None

In [117]:
def klik(b):
    print("Anda Memilih : ",dropdown.value)
    display(rekomendasi(dropdown.value))
button.on_click(klik)